## Targeted Password Guessing with Prompt Evolution for PassLLM

### Install dependencies

In [ ]:
!pip install -U transformers

In [3]:
!pip install -q openevolve transformers peft accelerate pyyaml flask nest_asyncio matplotlib

### Setup Kaggle and download data

In [4]:
import os
import sys

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
src_config = 'kaggle.json'
dst_config = os.path.join(kaggle_dir, 'kaggle.json')

if os.path.exists(src_config):
    import shutil
    if os.path.abspath(src_config) != os.path.abspath(dst_config):
        shutil.copy(src_config, dst_config)
    if os.name != 'nt':
        os.chmod(dst_config, 0o600)
    print("OK")
else:
    print('kaggle.json not found')

OK


In [ ]:
!apt-get update
!apt-get install -y unzip
!pip install -q kaggle
!mkdir -p /content/data
!chmod 777 /content/data
!kaggle competitions download -c password-guessing -p /content/data --force
!unzip -q -o /content/data/password-guessing.zip -d /content/data

### Environment constants and GPU check

In [6]:
import multiprocessing
try:
    multiprocessing.set_start_method("spawn", force=True)
except RuntimeError:
    pass

import os
import json
import torch

COMPETITION = "/content/data"
PATH_TRAIN = os.path.join(COMPETITION, "train.json")
PATH_ADAPTER_126_CSDN = os.path.join(COMPETITION, "126_csdn_disQwen0.5B", "126_csdn_disQwen0.5B")
BASE_MODEL_PASSLLM = "Qwen/Qwen2.5-0.5B-Instruct"

EXPERIMENT_DIR = os.path.join(os.getcwd(), "targeted_evolution_exp")
os.makedirs(EXPERIMENT_DIR, exist_ok=True)

os.environ["PASSLLM_COMPETITION"] = COMPETITION
os.environ["PASSLLM_TRAIN"] = PATH_TRAIN
os.environ["PASSLLM_ADAPTER_126_CSDN"] = PATH_ADAPTER_126_CSDN
os.environ["PASSLLM_BASE_MODEL"] = BASE_MODEL_PASSLLM
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print("COMPETITION:", COMPETITION)
print("PATH_TRAIN:", PATH_TRAIN)
print("PATH_ADAPTER_126_CSDN:", PATH_ADAPTER_126_CSDN)
print("EXPERIMENT_DIR:", EXPERIMENT_DIR)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU mode")

COMPETITION: /content/data
PATH_TRAIN: /content/data/train.json
PATH_ADAPTER_126_CSDN: /content/data/126_csdn_disQwen0.5B/126_csdn_disQwen0.5B
EXPERIMENT_DIR: /root/targeted_evolution_exp
GPU: NVIDIA GeForce RTX 5090


In [7]:
with open(PATH_TRAIN, "r", encoding="utf-8") as f:
    TRAIN_DATA = json.load(f)
if isinstance(TRAIN_DATA, dict):
    TRAIN_DATA = [TRAIN_DATA]
print("Train samples:", len(TRAIN_DATA))
if TRAIN_DATA:
    print("Example:", TRAIN_DATA[0])

Train samples: 16000
Example: {'Knowledge': {'Old password': '225654314'}, 'password': '225654314'}


### Password length distribution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

lengths = []
for item in TRAIN_DATA:
    pwd = item.get("password")
    if pwd and isinstance(pwd, str):
        lengths.append(len(pwd))

lengths = np.array(lengths)
mean = np.mean(lengths)
std = np.std(lengths)

print(f"Total: {len(lengths)}")
print(f"Mean: {mean:.2f}")
print(f"Std: {std:.2f}")
print(f"1σ: [{mean-std:.2f}, {mean+std:.2f}]")
print(f"2σ: [{mean-2*std:.2f}, {mean+2*std:.2f}]")

plt.hist(lengths, bins=21, alpha=0.7, edgecolor='black')
plt.axvline(mean, color='r', linestyle='dashed', label=f'mean={mean:.1f}')
plt.axvline(mean+std, color='orange', linestyle='dashed', label='+1σ')
plt.axvline(mean-std, color='orange', linestyle='dashed')
plt.axvline(mean+2*std, color='y', linestyle='dashed', label='+2σ')
plt.axvline(mean-2*std, color='y', linestyle='dashed')
plt.xlabel("Password length")
plt.ylabel("Frequency")
plt.legend()
plt.show()

### Evolution config and initial prompt

In [8]:
CONFIG_YAML = '''
llm:
  models:
    - name: openai/gpt-oss-120b
      api_base: https://api.zveno.ai/v1
      api_key: "YOUR_TOKEN"
      api_type: openai
      default_params:
        temperature: 1.0
        top_p: 0.95
        max_tokens: 10000
        presence_penalty: 0 # 0.5

evolution:
  generations: 100
  population_size: 2000
  elitism: 10

operators:
  mutation:
    type: llm
    llm_model: openai/gpt-oss-120b
    prompt_template: |
      You are improving a prompt for targeted password guessing.
      A prompt is always an instruction, not program code.
      Current prompt:
      """{program}"""
      Its crack rate on the training data is {score:.4f}.
      Propose a modified prompt that increases the crack rate.
      You may change wording, add constraints, remove ineffective parts, or introduce new heuristics.
      Output only the new prompt text, nothing else.
  crossover:
    type: llm
    llm_model: openai/gpt-oss-120b
    probability: 0.8
    prompt_template: |
      Combine two prompts to create a superior one.
      Prompt A (score {score_a:.4f}): """{program_a}"""
      Prompt B (score {score_b:.4f}): """{program_b}"""
      Output only the new prompt.

selection:
  type: tournament
  tournament_size: 5
'''

config_path = os.path.join(EXPERIMENT_DIR, "config_evolution.yaml")
with open(config_path, "w", encoding="utf-8") as f:
    f.write(CONFIG_YAML.strip())
print("Config saved:", config_path)

Config saved: /root/targeted_evolution_exp/config_evolution.yaml


In [9]:
INITIAL_PROMPT = """You are a password guessing model. Given the old password, generate one possible new password. Output only the password."""

initial_prompt_path = os.path.join(EXPERIMENT_DIR, "initial_prompt.txt")
with open(initial_prompt_path, "w", encoding="utf-8") as f:
    f.write(INITIAL_PROMPT)
print("Initial prompt saved:", initial_prompt_path)

Initial prompt saved: /root/targeted_evolution_exp/initial_prompt.txt


In [10]:
best_path = os.path.join(EXPERIMENT_DIR, "best_program.txt")
with open(initial_prompt_path, "r", encoding="utf-8") as src:
    initial_content = src.read()
with open(best_path, "w", encoding="utf-8") as dst:
    dst.write(initial_content)
print(f"Initial prompt written to {best_path}")

Initial prompt written to /root/targeted_evolution_exp/best_program.txt


### Core module passllm_core.py

In [17]:
%%writefile passllm_core.py
import os
import json
import torch
import subprocess
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

def get_gpu_memory():
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=memory.used,memory.total,utilization.gpu', '--format=csv,noheader,nounits'],
            capture_output=True, text=True, check=True
        )
        used, total, util = result.stdout.strip().split(', ')
        return f"GPU util: {util}%, VRAM: {used}/{total} MiB"
    except:
        return "GPU stats unavailable"

def load_model():
    base_model_name = os.environ.get("PASSLLM_BASE_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
    adapter_path = os.environ.get("PASSLLM_ADAPTER_126_CSDN", "")
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
    tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    )

    if adapter_path and os.path.exists(adapter_path):
        print(f"[INFO] Loading adapter from {adapter_path}")
        model = PeftModel.from_pretrained(model, adapter_path, is_trainable=False)
        model = model.merge_and_unload()
        print("[INFO] Adapter merged.")
    else:
        print("[WARNING] No adapter found.")

    model.eval()
    model = torch.compile(model, mode="reduce-overhead", fullgraph=False)
    return model, tokenizer

def generate_candidates_batch(old_passwords, prompt_template, model, tokenizer,
                              target_count=100, gen_multiplier=1.0, batch_size=32, show_progress=True):
    generation_count = int(target_count * gen_multiplier)
    results = [None] * len(old_passwords)

    prompts = []
    for old in old_passwords:
        messages = [
            {"role": "system", "content": prompt_template.strip()},
            {"role": "user", "content": f"Old password: {old}"}
        ]
        full_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompts.append(full_input)

    iterator = range(0, len(prompts), batch_size)
    if show_progress:
        from tqdm import tqdm
        iterator = tqdm(iterator, desc="Generating batches", unit="batch")

    for batch_idx, start in enumerate(iterator):
        end = min(start + batch_size, len(prompts))
        batch = prompts[start:end]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(model.device)

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=32,
                do_sample=True,
                temperature=0.9,
                top_p=0.95,
                num_return_sequences=generation_count,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                use_cache=True
            )

        for i in range(len(batch)):
            seqs = outputs[i * generation_count:(i+1) * generation_count]
            candidates = []
            for seq in seqs:
                gen_text = tokenizer.decode(seq[inputs['input_ids'][i].shape[0]:], skip_special_tokens=True)
                guess = gen_text.split("\n")[0].strip().strip('"').strip("'")
                if guess.lower().startswith("password:"):
                    guess = guess[9:].strip()
                if guess:
                    candidates.append(guess[:64])
            unique = []
            seen = set()
            for c in candidates:
                if c not in seen:
                    seen.add(c)
                    unique.append(c)
                if len(unique) == target_count:
                    break
            if len(unique) < target_count:
                import random, string
                chars = string.ascii_letters + string.digits + "!@#$%&*"
                old = old_passwords[start + i]
                for _ in range(target_count - len(unique)):
                    unique.append(old + random.choice(chars))
            results[start + i] = unique

        first_idx = start
        print(f"\nBatch {batch_idx}: old={old_passwords[first_idx]}")
        print(f"  First 20 candidates: {results[first_idx][:20]}")
        print(get_gpu_memory())
        print()

        if show_progress:
            iterator.set_postfix_str(get_gpu_memory())

    return results

def generate_candidates_single(old_password, prompt_template, model, tokenizer,
                               target_count=100, gen_multiplier=1.0):
    return generate_candidates_batch([old_password], prompt_template, model, tokenizer,
                                     target_count, gen_multiplier, batch_size=1, show_progress=False)[0]

def compute_crack_rate(valid_items, prompt_template, model, tokenizer,
                       sample_size=200, num_candidates=20, gen_multiplier=1.0, batch_size=15):
    indices = torch.randperm(len(valid_items))[:sample_size].tolist()
    sample_items = [valid_items[i] for i in indices]
    
    old_list = []
    target_list = []
    for item in sample_items:
        old = (item.get("Knowledge") or {}).get("Old password")
        target = item.get("password")
        if old and target:
            old_list.append(old)
            target_list.append(target)
    
    if not old_list:
        return 0.0
    
    candidates_batch = generate_candidates_batch(
        old_list, prompt_template, model, tokenizer,
        target_count=num_candidates, gen_multiplier=gen_multiplier,
        batch_size=batch_size, show_progress=True
    )
    
    correct = 0
    for i, cands in enumerate(candidates_batch):
        if target_list[i] in cands:
            correct += 1
    
    rate = correct / len(old_list)
    print(f"Evaluation: {correct}/{len(old_list)} correct. Crack Rate: {rate:.4f}")
    return rate

Overwriting passllm_core.py


### Evaluator script for OpenEvolve

In [18]:
os.environ["PASSLLM_EVAL_SAMPLE_SIZE"] = "200"
os.environ["PASSLLM_EVAL_NUM_CANDIDATES"] = "100"
os.environ["PASSLLM_EVAL_GEN_MULTIPLIER"] = "1.0"

In [19]:
%%writefile evaluator.py
import os
import sys
import json
import torch
sys.path.append(os.getcwd())
from passllm_core import load_model, compute_crack_rate

PATH_TRAIN = os.environ.get("PASSLLM_TRAIN", "")
if not PATH_TRAIN:
    print("Error: PASSLLM_TRAIN not set", file=sys.stderr)
    sys.exit(1)

SAMPLE_SIZE = int(os.environ.get("PASSLLM_EVAL_SAMPLE_SIZE", "200"))
NUM_CANDIDATES = int(os.environ.get("PASSLLM_EVAL_NUM_CANDIDATES", "100"))
GEN_MULTIPLIER = float(os.environ.get("PASSLLM_EVAL_GEN_MULTIPLIER", "1.0"))

MODEL, TOKENIZER = load_model()

def evaluate(program: str) -> dict:
    try:
        with open(PATH_TRAIN, "r", encoding="utf-8") as f:
            train_data = json.load(f)
        if isinstance(train_data, dict):
            train_data = [train_data]
        valid_items = []
        for item in train_data:
            old = (item.get("Knowledge") or {}).get("Old password")
            target = item.get("password")
            if old and target:
                valid_items.append(item)
        sample_size = min(SAMPLE_SIZE, len(valid_items))
        crack_rate = compute_crack_rate(
            valid_items,
            program,
            MODEL,
            TOKENIZER,
            sample_size=sample_size,
            num_candidates=NUM_CANDIDATES,
            gen_multiplier=GEN_MULTIPLIER
        )
        return {"combined_score": crack_rate}
    except Exception as e:
        print(f"Evaluation failed: {e}", file=sys.stderr)
        return {"combined_score": 0.0}

Overwriting evaluator.py


### Mutator gpt-oss-120b by ZvenoAI

In [20]:
from openai import OpenAI

zveno_token = "YOUR_TOKEN"

In [21]:
client = OpenAI(base_url="https://api.zveno.ai/v1", api_key=zveno_token)

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "Just say 'hello'"}],
    max_tokens=10000,
    temperature=1.0
)
print("Zveno API works:", response.choices[0].message.content)

Zveno API works: hello


### Run evolution (100 generations, 2000 population)

Evolution settings.

In [22]:
import nest_asyncio
nest_asyncio.apply()
os.environ["OPENAI_API_KEY"] = zveno_token
os.environ["OPENEVOLVE_NUM_WORKERS"] = "8"

The cycle.

In [23]:
from openevolve.api import run_evolution
from openevolve.config import load_config

config = load_config(config_path)
output_dir = os.path.join(EXPERIMENT_DIR, "openevolve_output")
result = run_evolution(
    initial_program=initial_prompt_path,
    evaluator="evaluator.py",
    config=config,
    iterations=100,
    output_dir=output_dir,
    cleanup=False,
)

print("\n=== Evolution finished ===")
print("Best combined_score (crack rate):", result.best_score)
print("Best metrics:", result.metrics)
if hasattr(result, "best_code") and result.best_code:
    print("\nBest prompt (first 500 chars):\n", result.best_code[:500])
if getattr(result, "output_dir", None):
    print("Output dir:", result.output_dir)

2026-04-21 08:20:01,486 - INFO - Logging to /root/targeted_evolution_exp/openevolve_output/logs/openevolve_20260421_082001.log
2026-04-21 08:20:01,489 - INFO - Set random seed to 42 for reproducibility
2026-04-21 08:20:01,500 - INFO - Initialized OpenAI LLM with model: openai/gpt-oss-120b
2026-04-21 08:20:01,501 - INFO - Initialized LLM ensemble with models: openai/gpt-oss-120b (weight: 1.00)
2026-04-21 08:20:01,512 - INFO - Initialized prompt sampler
2026-04-21 08:20:01,512 - INFO - Set custom templates: system=evaluator_system_message, user=None
2026-04-21 08:20:01,513 - INFO - Initialized program database with 0 programs
2026-04-21 08:20:03,176 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-21 08:20:03,176 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-04-21 08:20:03,296 - INFO - 

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

2026-04-21 08:20:03,675 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-21 08:20:03,793 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-0.5B-Instruct/7ae557604adf67be50417f59c2c2f167def9a775/tokenizer_config.json "HTTP/1.1 200 OK"
2026-04-21 08:20:03,907 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-0.5B-Instruct/7ae557604adf67be50417f59c2c2f167def9a775/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json: 0.00B [00:00, ?B/s]

2026-04-21 08:20:04,181 - INFO - HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-0.5B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-04-21 08:20:04,422 - INFO - HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-0.5B-Instruct/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-04-21 08:20:04,669 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/vocab.json "HTTP/1.1 307 Temporary Redirect"
2026-04-21 08:20:04,793 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-0.5B-Instruct/7ae557604adf67be50417f59c2c2f167def9a775/vocab.json "HTTP/1.1 200 OK"
2026-04-21 08:20:04,938 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-0.5B-Instruct/7ae557604adf67be50417f59c2c2f167def9a775/vocab.json "HTTP/1.1 200 OK"


...


Batch 0: old=19740512
  First 20 candidates: ['368569084', '86232620', 'houjianxin', 'oklicky0512', 'A19740512', 'chen', 'chen8263', 'woaini1314', 'Your new password is 138669126', '520woaini', '86975890', 'zhaoyue', 'abcd197405', 'ANGEL836', '6783574015', '888888', 'abcd19740512', 'admin', 'a83261900', 'laitaoyan']
GPU util: 98%, VRAM: 31846/32607 MiB


Batch 1: old=uuuu
  First 20 candidates: ['weidan1314', 'asdfgh1234', '8201297', 'asdfg123', '13842928017', 'wang', '872663941', 'caonima123', '13608724154', '19860218', '284673109', '123456789', 'abcd123', '87920269', 'cd082361', 'woainizhe1314', 'A123456789', 'liao', 'asdfgh123', '0032867941']
GPU util: 99%, VRAM: 31846/32607 MiB


Batch 2: old=6101
  First 20 candidates: ['woshiniba', 'ASDFGHJKL', '61019367', 'WOAINI', '8379082', 'lovemeng', 'likangxu', 'abcd123', 'andy2438', '27433893', 'lushan898', '610132', 'a6101', 'only8412', 'hangzi', '2963874', '6384291', 'ok', 'w6101', 'leothang']
GPU util: 99%, VRAM: 31846/32607 MiB


Batc

Generating batches: 100%|██████████| 14/14 [00:41<00:00,  2.93s/batch, GPU util: 73%, VRAM: 31846/32607 MiB]
2026-04-21 10:01:50,488 - INFO - New MAP-Elites cell occupied in island 4: {'complexity': 6, 'diversity': 6}
2026-04-21 10:01:50,488 - INFO - Iteration 100: Program f56121d6-4d0a-45d3-b2cd-197910fcc179 (parent: 9787b7e3-67b0-4f77-bc31-c3b357a574ed) completed in 65.60s
2026-04-21 10:01:50,488 - INFO - Metrics: combined_score=0.2950
2026-04-21 10:01:50,489 - INFO - Checkpoint interval reached at iteration 100
2026-04-21 10:01:50,490 - INFO - Island Status:
2026-04-21 10:01:50,490 - INFO -  * Island 0: 18 programs, best=0.3200, avg=0.2464, diversity=100.85, gen=20 (best: bea53ee5-a278-4516-8a2d-6d28bf75aff0)
2026-04-21 10:01:50,490 - INFO -    Island 1: 17 programs, best=0.2750, avg=0.2385, diversity=150.45, gen=20 (best: 09e38a5b-e5a6-4553-ba25-f04c878d7c38)
2026-04-21 10:01:50,491 - INFO -    Island 2: 18 programs, best=0.2850, avg=0.2364, diversity=5.38, gen=20 (best: 193e0618-0


Batch 0: old=19631002
  First 20 candidates: ['lichao018', '198649', '19640728', 'liujinbao', 'ok874321', '19631002', 'Asd1987', 'lewink1002', '19781204', 'qwertyuiop', 'wsxdyb84', '85245137', 'ABCD1987', 'ASDFGHJKL1', 'wzq8473', '8574377', 'wslxh86', 'ok', '1002', '10173104']
GPU util: 98%, VRAM: 31846/32607 MiB


Batch 1: old=222222
  First 20 candidates: ['222222.', '137668045', '221714', '261314', '016511418520', 'a101246', '13846036127', 'a123456789', 'ht147', 'love124', '222222', '0178460833', '123456', '107680189', 'wenlin8647', 'lwy870619', '19840607', 'qaz12wsx', '78401664', '07461141578']
GPU util: 98%, VRAM: 31846/32607 MiB


Batch 2: old=mmmmmmm
  First 20 candidates: ['123456789', 'asd8104', 'm19860104', '187648048a', 'ASDFGHJKL', 'A12345678', 'menghaiyou00', '68764904', '19800406', "I'm sorry to hear that you're having trouble with your old passw", '120676389', 'mcdwizhengjuan', 'm138764', 'asdfghjkl12', 'maters', 'mmmmmm', 'ljd1987', '14708654', '1.00787460', 'maoxiu850

2026-04-21 10:01:51,093 - INFO - Stopped process pool
2026-04-21 10:01:51,094 - INFO - Using tracked best program: bea53ee5-a278-4516-8a2d-6d28bf75aff0
2026-04-21 10:01:51,094 - INFO - Evolution complete. Best program has metrics: combined_score=0.3200
2026-04-21 10:01:51,094 - INFO - Saved best program to /root/targeted_evolution_exp/openevolve_output/best/best_program.txt with program info to /root/targeted_evolution_exp/openevolve_output/best/best_program_info.json



=== Evolution finished ===
Best combined_score (crack rate): 0.32
Best metrics: {'combined_score': 0.32}

Best prompt (first 500 chars):
 import sys, random, string

def generate_new_password(old_password):
    """
    Generate a plausible new password based on the old one.
    This simple heuristic creates a password of the same length
    by picking random printable characters (excluding whitespace).
    """
    printable = string.printable.rstrip()  # remove trailing whitespace characters
    return ''.join(random.choice(printable) for _ in range(len(old_password)))

def main():
    # Read the old password from standard input
 
Output dir: /root/targeted_evolution_exp/openevolve_output


Save the best prompt.

In [24]:
best_path = os.path.join(EXPERIMENT_DIR, "best_program.txt")
if hasattr(result, "best_code") and result.best_code:
    with open(best_path, "w", encoding="utf-8") as f:
        f.write(result.best_code)
    print("Best prompt saved:", best_path)

Best prompt saved: /root/targeted_evolution_exp/best_program.txt


### Evaluate best prompt on train sample

In [26]:
import torch
import json
import random
from tqdm import tqdm
from passllm_core import load_model, generate_candidates_batch

In [31]:
with open(best_path, "r") as f:
    prompt_template = f.read().strip()

model, tokenizer = load_model()

2026-04-21 10:07:05,056 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-21 10:07:05,187 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-0.5B-Instruct/7ae557604adf67be50417f59c2c2f167def9a775/config.json "HTTP/1.1 200 OK"
2026-04-21 10:07:05,433 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-21 10:07:05,551 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-0.5B-Instruct/7ae557604adf67be50417f59c2c2f167def9a775/tokenizer_config.json "HTTP/1.1 200 OK"
2026-04-21 10:07:05,794 - INFO - HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-0.5B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-04-21 10:07:06,037 - INFO - HTTP Request: GET https://h

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

2026-04-21 10:07:07,533 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-21 10:07:07,647 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-0.5B-Instruct/7ae557604adf67be50417f59c2c2f167def9a775/generation_config.json "HTTP/1.1 200 OK"
2026-04-21 10:07:07,885 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/custom_generate/generate.py "HTTP/1.1 404 Not Found"


[INFO] Loading adapter from /content/data/126_csdn_disQwen0.5B/126_csdn_disQwen0.5B
[INFO] Adapter merged.


In [32]:
best_prompt_path = os.path.join(EXPERIMENT_DIR, "best_program.txt")
with open(best_prompt_path, "r", encoding="utf-8") as f:
    prompt_template = f.read().strip()
print("Loaded best prompt:", prompt_template[:500])

Loaded best prompt: import sys, random, string

def generate_new_password(old_password):
    """
    Generate a plausible new password based on the old one.
    This simple heuristic creates a password of the same length
    by picking random printable characters (excluding whitespace).
    """
    printable = string.printable.rstrip()  # remove trailing whitespace characters
    return ''.join(random.choice(printable) for _ in range(len(old_password)))

def main():
    # Read the old password from standard input
 


In [33]:
valid_items = []
for item in TRAIN_DATA:
    old = (item.get("Knowledge") or {}).get("Old password")
    target = item.get("password")
    if old and target:
        valid_items.append({"old": old, "target": target})

random.seed(42)
valid_items = random.sample(valid_items, min(500, len(valid_items)))

sample_size = 500
indices = list(range(sample_size))
sample_items = [valid_items[i] for i in indices]

old_list = [it["old"] for it in sample_items]
target_list = [it["target"] for it in sample_items]

candidates_list = generate_candidates_batch(old_list, prompt_template, model, tokenizer,
                                            target_count=100, batch_size=16)

correct = 0
for i, cands in enumerate(candidates_list):
    if target_list[i] in cands:
        correct += 1

crack_rate = correct / sample_size
print(f"Crack rate on {sample_size} samples: {crack_rate:.4f} ({crack_rate*100:.2f}%)")

Generating batches:   3%|▎         | 1/32 [00:06<03:13,  6.24s/batch, GPU util: 99%, VRAM: 24567/32607 MiB]


Batch 0: old=564445087
  First 20 candidates: ['a123456', '9812843520', '2935341', '230976153', '564445087w', '23192710', '521yuan', '.sdfgh123123', '13124905830', '123456789', '123456', '2130975', 'cxj198239', '564445087', 'b1988823', 'bairule', '.23812936', '8179659', '...', 'woainijibenzi']
GPU util: 99%, VRAM: 24567/32607 MiB



Generating batches:   6%|▋         | 2/32 [00:12<03:08,  6.27s/batch, GPU util: 99%, VRAM: 24581/32607 MiB]


Batch 1: old=77777777
  First 20 candidates: ['7382594', 'woaini123', '0123456789', '49301588', '789456123', '582369041', 'shumei836', '58140982', '051286zl', '71534896420', '777liuje', '123456', '7513689', 'shiren123', '.2538046', '89513240', '08592416', '104105792', '13524980617', '86294521']
GPU util: 99%, VRAM: 24581/32607 MiB



Generating batches:   9%|▉         | 3/32 [00:18<03:01,  6.27s/batch, GPU util: 99%, VRAM: 24581/32607 MiB]

...


Batch 29: old=1042614606
  First 20 candidates: ['5889238', 'ABCDEFGHIJKLMN', '87534115', 'shuaiwo334', 'a5239808', '8763945456', 'woailexiuzhengme', 'woainiduge', '.', '.38795219', '123qweasd', '19870306', '578893076', 'wenguojia_870', '..abcdefghjklmn', '13972825891', '1987210005', 'Asdfghjklmn', '13874543994', '135792468']
GPU util: 99%, VRAM: 24811/32607 MiB



Generating batches:  97%|█████████▋| 31/32 [03:15<00:06,  6.30s/batch, GPU util: 99%, VRAM: 24811/32607 MiB]


Batch 30: old=19670420
  First 20 candidates: ['000000', '13587512713', '582093105', 'This script generates a new password using the principle of rand', '420zyf', "Here's the script that generates a new password based on the old", "Here's an expanded version of the script that demonstrates how t", '5585932', '```python', 'A8531656', '41181936', '01182519', '19830915', 'luokai5812', '13528673231', '19850503', '....039548', 'AC854375', '45346869', '19831225']
GPU util: 99%, VRAM: 24811/32607 MiB



Generating batches: 100%|██████████| 32/32 [03:16<00:00,  6.16s/batch, GPU util: 96%, VRAM: 24821/32607 MiB]


Batch 31: old=1984417910
  First 20 candidates: ['654123456789', 'lxw530362', '4253065', 'lovezihang131', '/16235470', '6524816', '643854123', '.ok824321520', '.2586381615', '198441791', '.op369520', '05230980', '8569632', '26318513', '123456', 'The program has generated and returned a new password. Can you p', "Here's the modified script with comments added for clarity:", '13652597401', '.5221314.', 'asdfghjklmn']
GPU util: 96%, VRAM: 24821/32607 MiB

Crack rate on 500 samples: 0.2040 (20.40%)


In [34]:
prompt_template

'import sys, random, string\n\ndef generate_new_password(old_password):\n    """\n    Generate a plausible new password based on the old one.\n    This simple heuristic creates a password of the same length\n    by picking random printable characters (excluding whitespace).\n    """\n    printable = string.printable.rstrip()  # remove trailing whitespace characters\n    return \'\'.join(random.choice(printable) for _ in range(len(old_password)))\n\ndef main():\n    # Read the old password from standard input\n    old = sys.stdin.read().strip()\n    if not old:\n        # If no input is provided, output an empty line (still satisfies the spec)\n        print("")\n        return\n\n    new_password = generate_new_password(old)\n    # Output only the new password as required\n    print(new_password)\n\nif __name__ == "__main__":\n    main()'

### Final inference on test set and submission

In [35]:
TEST_PATH = "/content/data/test.json"
with open(TEST_PATH, "r") as f:
    test_data = json.load(f)
old_passwords = [item["Knowledge"]["Old password"] for item in test_data]
print(f"Test samples: {len(old_passwords)}")

Test samples: 8000


In [37]:
print("Starting inference on test set...")
all_candidates = generate_candidates_batch(
    old_passwords, prompt_template, model, tokenizer,
    target_count=100, gen_multiplier=1.0, batch_size=15, show_progress=True
)

Starting inference on test set...


Generating batches:   0%|          | 1/534 [00:05<51:56,  5.85s/batch, GPU util: 99%, VRAM: 24821/32607 MiB]


Batch 0: old=eye88gogO
  First 20 candidates: ['123456', '13927474501', '03496712', '0123456789', 'e123456789', 'E5874372', '0413296588', 'eat3690472', '63157954', 'E547296410', '67510942', '1234567890', '071123140', 'est2009', '0712658954', '09657417032', '61234567', "Here's your generated new password, with each character selected", '09163248a', '03572411']
GPU util: 99%, VRAM: 24821/32607 MiB



Generating batches:   0%|          | 2/534 [00:11<51:50,  5.85s/batch, GPU util: 99%, VRAM: 24821/32607 MiB]


Batch 1: old=max14
  First 20 candidates: ['603947520', 'Asdfghjklmn0987', '2561809a', 'ABCxxx2008', '63207458', 'A857306', 'weilu0618', '02030579', 'Amaxin500', '.527802968', '/cslj0605', 'wengaixueyi', 'xuefei520', '..893067554', '/second/', '80623594', '890237asdf', '.520867', 'a123456789', 'mao520']
GPU util: 99%, VRAM: 24821/32607 MiB



...

Generating batches: 100%|█████████▉| 533/534 [52:29<00:05,  5.91s/batch, GPU util: 99%, VRAM: 22013/32607 MiB]


Batch 532: old=yyyyyyy
  First 20 candidates: ['west123456', 'xvly7459', '021687430', 'youngxia01', 'yangmei507', '829480567', 'yyyyyyyy', 'yuedalin', '2864103879', 'yuanxiao1987', '645123499', '5201314liu', 'y3125754', 'yyyyyyy', '8459611', '2168950', 'y123456789', 'yubeng0123', '15863421709', '15906483472']
GPU util: 99%, VRAM: 22013/32607 MiB



Generating batches: 100%|██████████| 534/534 [52:31<00:00,  5.90s/batch, GPU util: 97%, VRAM: 22023/32607 MiB]


Batch 533: old=9893
  First 20 candidates: ['989374162', '07514886291', '123456789', '989345678', 'wangchen', '0216', 'liushao', 'shuailixie', '9893a', '9893456789', '66510420', '617046521', 'abc7654321', '0123456789', '942675', '9893waka', 'qwer1234', '5260174', 'asd12345678', '0415zy']
GPU util: 97%, VRAM: 22023/32607 MiB



NameError: name 'get_gpu_memory' is not defined

In [40]:
import pandas as pd

rows = [[i] + cands for i, cands in enumerate(all_candidates)]
columns = ["id"] + [f"password{j+1}" for j in range(100)]
df = pd.DataFrame(rows, columns=columns)
df.to_csv("submission.csv", index=False)
print("Submission saved: submission.csv")

Submission saved: submission.csv


In [39]:
!pip install -q pandas

In [41]:
!kaggle competitions submit -c password-guessing -f submission.csv -m "Evolution 100gen 2000pop"
print("Submitted.")

100%|███████████████████████████████████████| 8.37M/8.37M [00:11<00:00, 748kB/s]
Successfully submitted to P455F1nd3rSubmitted.
